In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
silver_orders = spark.table("workspace.final_project.silver_orders")
silver_items = spark.table("workspace.final_project.silver_items")
silver_suppliers = spark.table("workspace.final_project.silver_suppliers")
silver_incidents = spark.table("workspace.final_project.silver_incidents")

In [0]:
dim_suppliers = (
    silver_suppliers
    .select(
        "supplier_id",
        "supplier_name",
        "city",
        "country",
        "country_code"
    )
)

dim_suppliers.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_dim_suppliers"
)

In [0]:
# a preparer peut etre dès la couche silver
fact_order_items = (
    silver_items
    .join(silver_orders, "order_id")
    .withColumn("order_month", F.trunc("order_date", "month"))
    .withColumn("spend", F.col("quantity") * F.col("unit_price"))
)

fact_order_items.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.final_project.gold_fact_order_items")

In [0]:
fact_incidents = (
    silver_incidents
    .join(
        silver_orders.select("order_id", "supplier_id", "order_date"),
        "order_id",
        "left"
    )
    .withColumn("incident_month", F.trunc("incident_date", "month"))
)

fact_incidents.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_fact_incidents"
)

In [0]:
orders_monthly = (
    fact_order_items
    .groupBy(
        "supplier_id",
        F.year("order_date").alias("year"),
        F.month("order_date").alias("month")
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("spend").alias("total_spend"),
        F.avg("spend").alias("avg_order_value"),

        F.avg(
            F.when(
                F.col("delivery_date_actual") <= F.col("delivery_date_expected"),
                1
            ).otherwise(0)
        ).alias("on_time_delivery_rate"),

        F.avg(
            F.datediff(
                "delivery_date_actual",
                "delivery_date_expected"
            )
        ).alias("avg_delay_days")
    )
)

In [0]:

incidents_monthly = (
    fact_incidents
    .withColumn("year", F.year("incident_date"))
    .withColumn("month", F.month("incident_date"))
    .groupBy("supplier_id", "year", "month")
    .agg(F.count("*").alias("incident_count"))
)

In [0]:
supplier_monthly = (
    orders_monthly
    .join(
        incidents_monthly,
        ["supplier_id", "year", "month"],
        "left"
    )
    .fillna({"incident_count": 0})
)

supplier_monthly = supplier_monthly.withColumn(
    "incident_rate",
    F.col("incident_count") / F.col("total_orders")
)

In [0]:
supplier_monthly = (
    orders_monthly
    .join(
        incidents_monthly,
        ["supplier_id", "year", "month"],
        "left"
    )
    .fillna({"incident_count": 0})
)

supplier_monthly = supplier_monthly.withColumn(
    "incident_rate",
    F.col("incident_count") / F.col("total_orders")
)


In [0]:
display(supplier_monthly)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Exclure le supplier 
supplier_monthly_filtered = supplier_monthly.filter(
    ~F.col("supplier_id").isin(29)
)
display(supplier_monthly_filtered)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
window_3m = (
    Window
    .partitionBy("supplier_id")
    .orderBy("year", "month")
    .rowsBetween(-2, 0)
)

supplier_monthly = (
    supplier_monthly
    .withColumn(
        "rolling_orders_3m",
        F.sum("total_orders").over(window_3m)
    )
    .withColumn(
        "rolling_spend_3m",
        F.sum("total_spend").over(window_3m)
    )
    .withColumn(
        "rolling_incident_rate_3m",
        F.avg("incident_rate").over(window_3m)
    )
    .withColumn(
        "rolling_delivery_rate_3m",
        F.avg("on_time_delivery_rate").over(window_3m)
    )
)

In [0]:
supplier_monthly.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_supplier_performance_monthly"
)

In [0]:
supplier_monthly_filtered = supplier_monthly.filter(
    ~F.col("supplier_id").isin(29)
)
display(supplier_monthly_filtered)

Databricks visualization. Run in Databricks to view.

In [0]:
supplier_yearly = (
    fact_order_items
    .groupBy(
        "supplier_id",
        F.year("order_date").alias("year")
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("spend").alias("total_spend"),
        F.avg("spend").alias("avg_order_value")
    )
)

In [0]:
incidents_yearly = (
    fact_incidents
    .withColumn("year", F.year("incident_date"))  
    .groupBy("supplier_id", "year")
    .agg(
        F.count("*").alias("incident_count"),
        F.sum(F.when(F.col("severity")=="Critical", 1).otherwise(0)).alias("nb_critical_incidents"),
        F.sum(F.when(F.col("severity")=="High", 1).otherwise(0)).alias("nb_high_incidents"),
        F.sum(F.when(F.col("severity")=="Medium", 1).otherwise(0)).alias("nb_medium_incidents"),
        F.sum(F.when(F.col("severity")=="Low", 1).otherwise(0)).alias("nb_low_incidents")
    )
)

supplier_yearly = (
    supplier_yearly
    .join(
        incidents_yearly,
        ["supplier_id","year"],
        "left"
    )
    .fillna({
        "incident_count": 0,
        "nb_critical_incidents": 0,
        "nb_high_incidents": 0,
        "nb_medium_incidents": 0,
        "nb_low_incidents": 0
    })
)

In [0]:
# Nombre total d'incidents par fournisseur dans le temps
display(supplier_yearly)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Empilement des incidents critiques, élevés et moyens
display(
    supplier_yearly
    .groupBy("supplier_id", "year")
    .agg(
        F.sum("nb_critical_incidents").alias("critical"),
        F.sum("nb_high_incidents").alias("high"),
        F.sum("nb_medium_incidents").alias("medium"),
        F.sum("nb_low_incidents").alias("low")
    )
    .orderBy("year")
)

Databricks visualization. Run in Databricks to view.

In [0]:
category_metrics = (
    fact_order_items
    .groupBy(
        "product_category",
        F.year("order_date").alias("year")
    )
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_quantity"),
        F.sum("spend").alias("total_spend"),
        F.avg("unit_price").alias("avg_unit_price"),
        F.countDistinct("supplier_id").alias("supplier_count")
    )
)

category_metrics.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_category_metrics_yearly"
)

In [0]:
# Bar chart — total spend par catégorie et année
display(
    category_metrics.select("product_category", "year", "total_spend")
)

# Line chart — évolution du nombre de commandes par catégorie
display(
    category_metrics.select("product_category", "year", "total_orders")
)

# Bar chart ou pie chart — nombre de fournisseurs par catégorie
display(
    category_metrics.select("product_category", "year", "supplier_count")
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:

# -----------------------------------------
# 1️ Agrégation annuelle à partir des faits Gold
# -----------------------------------------

supplier_yearly = (
    spark.table("workspace.final_project.gold_supplier_performance_monthly")
    .groupBy("supplier_id", "year")
    .agg(
        F.sum("total_orders").alias("total_orders"),
        F.sum("total_spend").alias("total_spend"),
        F.avg("on_time_delivery_rate").alias("on_time_delivery_rate"),
        F.sum("incident_count").alias("incident_count")
    )
)

# -----------------------------------------
# 2️ Calcul des scores
# -----------------------------------------

# Delivery score (% commandes à temps)
supplier_scores = supplier_yearly.withColumn(
    "delivery_score",
    F.col("on_time_delivery_rate") * 100
)

# FIX IMPORTANT : incident_score borné entre 0 et 100
# incident_count peut être > total_orders
incident_rate = F.col("incident_count") / F.col("total_orders")

supplier_scores = supplier_scores.withColumn(
    "incident_rate_capped",
    F.when(incident_rate > 1, 1).otherwise(incident_rate)
)

supplier_scores = supplier_scores.withColumn(
    "incident_score",
    (1 - F.col("incident_rate_capped")) * 100
)

# Activity score pondéré par spend
max_spend = supplier_scores.agg(F.max("total_spend")).collect()[0][0]

supplier_scores = supplier_scores.withColumn(
    "activity_score",
    (F.col("total_spend") / F.lit(max_spend)) * 100
)

# Score global pondéré
supplier_scores = supplier_scores.withColumn(
    "supplier_score",
    0.5 * F.col("delivery_score") +
    0.3 * F.col("incident_score") +
    0.2 * F.col("activity_score")
)

# -----------------------------------------
# 3️ Classement des fournisseurs
# -----------------------------------------
window_rank = Window.orderBy(F.col("supplier_score").desc())

supplier_scores = supplier_scores.withColumn(
    "supplier_rank",
    F.rank().over(window_rank)
)

# -----------------------------------------
# 4️ Écriture dans Gold
# -----------------------------------------
supplier_scores.write.mode("overwrite").saveAsTable(
    "workspace.final_project.gold_supplier_scores_yearly"
)

# -----------------------------------------
# 5️ Display / KPI 
# -----------------------------------------

# 5a — Classement top 20 fournisseurs
display(
    supplier_scores.select("supplier_rank", "supplier_id", "supplier_score")
    .orderBy(F.col("supplier_score").desc())
    .limit(20)
)

# 5b — Composition des scores pour top 10 fournisseurs (bar chart ou radar chart)
display(
    supplier_scores.select(
        "supplier_id", "delivery_score", "incident_score", "activity_score", "supplier_score"
    ).orderBy(F.col("supplier_score").desc()).limit(10)
)

# 5c — Distribution des scores (histogramme)
display(
    supplier_scores.select("supplier_score")
)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.